In [ ]:
# ===================================================================
# Surveillance anomaly detection on UCF-Crime, frame-level AUC.
#
# WHAT THIS IS
#   A benchmark measurement. Weakly-supervised multiple-instance
#   learning (Sultani et al. style) on frozen ImageNet ResNet-50
#   features: training uses only VIDEO-level labels, and the reported
#   number is frame-level ROC AUC on the held-out test videos against
#   the OFFICIAL per-frame temporal annotations.
#
# WHAT THIS IS NOT
#   This is NOT a deployable theft detector. It is not a detector of
#   anything. It produces a per-frame anomaly score whose ranking
#   quality is summarised by one AUC on one curated benchmark whose
#   "anomalous" videos were selected and trimmed by annotators. The
#   false-positive rate at fixed recall is reported precisely because
#   AUC hides how unusable such a score is at any operating point.
#   Nothing here should be pointed at a real person.
# ===================================================================
import os, sys, json, glob, time, collections, urllib.request, random
import numpy as np
print("python", sys.version)
import torch, torchvision
print("torch", torch.__version__, "torchvision", torchvision.__version__, "cuda", torch.cuda.is_available())
from PIL import Image

ROOT = "/kaggle/input"
print("TOP LEVEL:", sorted(os.listdir(ROOT)))

ext = collections.Counter(); bydir = collections.Counter(); samples = []
ndirs = 0
for dp, dn, fn in os.walk(ROOT):          # NO depth cap
    ndirs += 1
    if fn:
        bydir[os.path.relpath(dp, ROOT)] += len(fn)
    for f in fn:
        ext[os.path.splitext(f)[1].lower()] += 1
        if len(samples) < 20:
            samples.append(os.path.join(os.path.relpath(dp, ROOT), f))
print("dirs walked:", ndirs)
print("extensions:", ext.most_common(10))
print("dirs by file count:")
for d, c in bydir.most_common(40):
    print("  %8d %s" % (c, d))
print("sample real paths:")
for s in samples:
    print("   ", s)

In [ ]:
# ---- index the frame dataset: Train/<class>/<video>_<frameidx>.png ----
def find_split(name):
    """Find a directory called `name` at ANY depth under /kaggle/input.

    Mount depth is not constant between Kaggle datasets, so this walks
    instead of guessing a depth. It prefers the candidate with the most
    class subfolders holding images.
    """
    hits = []
    for dp, dn, fn in os.walk(ROOT):
        if os.path.basename(dp) == name:
            subs = [d for d in os.listdir(dp) if os.path.isdir(os.path.join(dp, d))]
            score = 0
            for d in subs:
                try:
                    score += sum(1 for f in os.listdir(os.path.join(dp, d))[:50]
                                 if os.path.splitext(os.path.basename(f))[1].lower() in IMG_EXT)
                except OSError:
                    pass
            hits.append((len(subs), score, dp))
    hits.sort(reverse=True)
    print("  candidates for %r:" % name, [(h[0], h[1], h[2]) for h in hits])
    return hits[0][2] if hits else None

IMG_EXT = (".png", ".jpg", ".jpeg")
TR_DIR, TE_DIR = find_split("Train"), find_split("Test")
print("Train dir:", TR_DIR)
print("Test  dir:", TE_DIR)
assert TR_DIR and TE_DIR, "Train/ and Test/ frame folders not found"

def index(d):
    """-> {video_name: {'cls':..., 'frames':[(idx, path)...]}}"""
    vids = {}
    for cls in sorted(os.listdir(d)):
        cd = os.path.join(d, cls)
        if not os.path.isdir(cd):
            continue
        for f in os.listdir(cd):
            bn = os.path.basename(f)                     # BASENAME only
            stem, e = os.path.splitext(bn)
            if e.lower() not in IMG_EXT:
                continue
            if "_" not in stem:
                continue
            vid, _, idx = stem.rpartition("_")
            if not idx.isdigit():
                continue
            v = vids.setdefault(vid, {"cls": cls, "frames": []})
            v["frames"].append((int(idx), os.path.join(cd, bn)))
    for v in vids.values():
        v["frames"].sort()
    return vids

t0 = time.time()
TRV = index(TR_DIR)
TEV = index(TE_DIR)
print("indexed in %.0fs" % (time.time() - t0))
print("train videos:", len(TRV), "frames:", sum(len(v["frames"]) for v in TRV.values()))
print("test  videos:", len(TEV), "frames:", sum(len(v["frames"]) for v in TEV.values()))
print("train classes:", sorted(set(v["cls"] for v in TRV.values())))
print("test  classes:", sorted(set(v["cls"] for v in TEV.values())))
for k in list(TEV)[:5]:
    fr = TEV[k]["frames"]
    print("  %-24s cls=%-18s n=%4d idx[0:5]=%s max=%d" % (k, TEV[k]["cls"], len(fr),
          [i for i, _ in fr[:5]], fr[-1][0]))
im0 = Image.open(TEV[list(TEV)[0]]["frames"][0][1])
print("frame size/mode:", im0.size, im0.mode)
assert len(TRV) > 500 and len(TEV) > 100, "too few videos indexed"

overlap = set(TRV) & set(TEV)
print("train/test video-name overlap:", len(overlap), sorted(overlap)[:5])
assert len(overlap) == 0, "%d videos appear in both Train and Test - the split is not held out" % len(overlap)

def is_normal(cls):
    return "normal" in cls.lower()
print("train normal videos:", sum(1 for v in TRV.values() if is_normal(v["cls"])),
      "anomalous:", sum(1 for v in TRV.values() if not is_normal(v["cls"])))
print("test  normal videos:", sum(1 for v in TEV.values() if is_normal(v["cls"])),
      "anomalous:", sum(1 for v in TEV.values() if not is_normal(v["cls"])))

In [ ]:
# ---- OFFICIAL frame-level temporal annotations for the test videos ----
# Without these, frame labels would have to be faked from the video label,
# which would make every frame of an anomalous video "anomalous" and the
# resulting AUC meaningless. The run fails rather than do that.
URLS = [
 # Verified reachable from a Kaggle kernel on 2026-10-04, 290 lines, format
 # "Abuse028_x264.mp4  Abuse  165  240  -1  -1". The others are kept as
 # fallbacks; most 404 and crcv.ucf.edu fails TLS verification from Kaggle.
 "https://raw.githubusercontent.com/WaqasSultani/AnomalyDetectionCVPR2018/master/Temporal_Anomaly_Annotation.txt",
 "https://raw.githubusercontent.com/tianyu0207/RTFM/main/list/Temporal_Anomaly_Annotation_for_Testing_Videos.txt",
 "https://raw.githubusercontent.com/Roc-Ng/DeepMIL/master/list/Temporal_Anomaly_Annotation_for_Testing_Videos.txt",
 "https://raw.githubusercontent.com/Roc-Ng/XDVioDet/master/list/Temporal_Anomaly_Annotation_for_Testing_Videos.txt",
 "https://raw.githubusercontent.com/WaqasSultani/AnomalyDetectionCVPR2018/master/Temporal_Anomaly_Annotation.txt",
 "https://raw.githubusercontent.com/ktr-hubrt/WSAL/master/data/Temporal_Anomaly_Annotation_for_Testing_Videos.txt",
 "https://raw.githubusercontent.com/junha-l/UCF-Crime/main/Temporal_Anomaly_Annotation_for_Testing_Videos.txt",
 "https://raw.githubusercontent.com/jx-zhong-for-academic-purpose/GCN-Anomaly-Detection/master/Temporal_Anomaly_Annotation_for_Testing_Videos.txt",
]

def parse_anno(text):
    """lines like: Abuse028_x264.mp4  Abuse  165  240  -1  -1"""
    out = {}
    for line in text.replace(",", " ").strip().splitlines():
        p = line.split()
        if len(p) < 4:
            continue
        name = os.path.splitext(os.path.basename(p[0]))[0]
        nums = [x for x in p[2:] if x.lstrip("-").isdigit()]
        if len(nums) < 2:
            continue
        ivs = []
        for i in range(0, len(nums) - 1, 2):
            s, e = int(nums[i]), int(nums[i + 1])
            if s >= 0 and e > s:
                ivs.append((s, e))
        out[name] = {"cls": p[1], "intervals": ivs}
    return out

ANNO = None
SRC = None
for u in URLS:
    try:
        txt = urllib.request.urlopen(u, timeout=45).read().decode("utf-8", "replace")
        a = parse_anno(txt)
        print("fetched %s -> %d parsed video entries" % (u, len(a)))
        if len(a) >= 250:
            ANNO, SRC = a, u
            print("\n".join(txt.strip().splitlines()[:5]))
            break
    except Exception as e:
        print("FAIL", u, type(e).__name__, e)
assert ANNO is not None, "could not obtain the official UCF-Crime temporal annotations; refusing to fabricate frame labels"
print("annotation source:", SRC, "| videos:", len(ANNO))
print("videos with >=1 anomalous interval:", sum(1 for v in ANNO.values() if v["intervals"]))

covered = [v for v in TEV if v in ANNO]
print("test videos covered by annotations: %d / %d" % (len(covered), len(TEV)))
miss = [v for v in TEV if v not in ANNO]
print("uncovered examples:", miss[:10])
assert len(covered) >= 0.8 * len(TEV), "only %d/%d test videos have official frame annotations" % (len(covered), len(TEV))

# sanity: annotated-normal videos should be the ones in the Normal class folder
agree = sum(1 for v in covered if (len(ANNO[v]["intervals"]) == 0) == is_normal(TEV[v]["cls"]))
print("agreement between folder class and annotation having intervals: %d / %d" % (agree, len(covered)))
assert agree >= 0.9 * len(covered), "annotations and folder labels disagree on %d videos - wrong annotation file" % (len(covered) - agree)

TEST_VIDS = sorted(covered)

In [ ]:
# ---- frozen ImageNet ResNet-50 features ----
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "GPU not attached"
WT = ResNet50_Weights.IMAGENET1K_V2
backbone = resnet50(weights=WT)
backbone.fc = torch.nn.Identity()
backbone = backbone.eval().to(dev)
print("backbone:", WT)

TF = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class Frames(Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        try:
            im = Image.open(self.paths[i]).convert("RGB")
            return TF(im), 1
        except Exception:
            return torch.zeros(3, 224, 224), 0

def feats(paths, tag):
    dl = DataLoader(Frames(paths), batch_size=256, num_workers=4, shuffle=False)
    F, OK = [], []
    t0 = time.time()
    with torch.inference_mode():
        for bi, (x, ok) in enumerate(dl):
            with torch.autocast("cuda", dtype=torch.float16):
                f = backbone(x.to(dev, non_blocking=True))
            F.append(f.half().cpu().numpy())
            OK.append(ok.numpy())
            if bi % 100 == 0:
                print("  %s %d/%d %.0fs" % (tag, bi, len(dl), time.time() - t0), flush=True)
    F = np.concatenate(F)
    OK = np.concatenate(OK).astype(bool)
    print("   dtype", F.dtype, "bytes", F.nbytes)
    print("%s: %s ok=%d/%d in %.0fs" % (tag, F.shape, OK.sum(), len(OK), time.time() - t0))
    return F, OK

MAX_TRAIN_FRAMES_PER_VIDEO = 128

train_paths, train_owner = [], []
for v in sorted(TRV):
    fr = TRV[v]["frames"]
    if len(fr) > MAX_TRAIN_FRAMES_PER_VIDEO:
        sel = np.linspace(0, len(fr) - 1, MAX_TRAIN_FRAMES_PER_VIDEO).round().astype(int)
        fr = [fr[i] for i in sel]
    for idx, p in fr:
        train_paths.append(p)
        train_owner.append(v)
print("train frames to embed:", len(train_paths))

test_paths, test_owner, test_fidx = [], [], []
for v in TEST_VIDS:
    for idx, p in TEV[v]["frames"]:
        test_paths.append(p)
        test_owner.append(v)
        test_fidx.append(idx)
print("test frames to embed:", len(test_paths))
assert len(test_paths) >= 20000, "only %d test frames" % len(test_paths)

Ftr, OKtr = feats(train_paths, "train")
Fte, OKte = feats(test_paths, "test")
assert OKtr.mean() > 0.95 and OKte.mean() > 0.95, "too many unreadable frames"
assert float(Fte[:2000].astype(np.float32).std()) > 1e-6, "features are constant - extraction is broken"

In [ ]:
# ---- 32-segment bags per video ----
NSEG = 32

def bags(F, owner, vids):
    by = collections.defaultdict(list)
    for i, v in enumerate(owner):
        by[v].append(i)
    B, keep = {}, []
    for v in vids:
        ii = by.get(v, [])
        if len(ii) < NSEG:
            continue
        ii = np.array(ii)
        edges = np.linspace(0, len(ii), NSEG + 1).round().astype(int)
        seg = np.stack([F[ii[edges[k]:edges[k + 1]]].astype(np.float32).mean(axis=0) for k in range(NSEG)])
        n = np.linalg.norm(seg, axis=1, keepdims=True)
        B[v] = (seg / np.maximum(n, 1e-8)).astype(np.float32)
        keep.append(v)
    return B, keep

TR_ALL = sorted(TRV)
Btr, tr_keep = bags(Ftr, train_owner, TR_ALL)
Bte, te_keep = bags(Fte, test_owner, TEST_VIDS)
print("train bags:", len(tr_keep), "test bags:", len(te_keep))
assert len(tr_keep) > 400 and len(te_keep) > 100

random.seed(0)
anom = [v for v in tr_keep if not is_normal(TRV[v]["cls"])]
norm = [v for v in tr_keep if is_normal(TRV[v]["cls"])]
random.shuffle(anom); random.shuffle(norm)
na, nn = max(1, len(anom) // 10), max(1, len(norm) // 10)
VAL = anom[:na] + norm[:nn]
FIT_A, FIT_N = anom[na:], norm[nn:]
print("fit: %d anomalous / %d normal | val(held out from fit, video-level): %d" % (len(FIT_A), len(FIT_N), len(VAL)))
assert len(FIT_A) > 50 and len(FIT_N) > 50

In [ ]:
# ---- MIL ranking model ----
D = Btr[tr_keep[0]].shape[1]

class MIL(torch.nn.Module):
    def __init__(self, d):
        super().__init__()
        self.f = torch.nn.Sequential(
            torch.nn.Linear(d, 512), torch.nn.ReLU(), torch.nn.Dropout(0.6),
            torch.nn.Linear(512, 32), torch.nn.ReLU(), torch.nn.Dropout(0.6),
            torch.nn.Linear(32, 1))
    def forward(self, x):
        return torch.sigmoid(self.f(x)).squeeze(-1)

from sklearn.metrics import roc_auc_score, roc_curve

def video_level_auc(net, vids, labels):
    net.eval()
    s = []
    with torch.inference_mode():
        for v in vids:
            x = torch.from_numpy(Btr[v]).to(dev)
            s.append(float(net(x).max()))
    net.train()
    return roc_auc_score(labels, s)

val_lab = [0 if is_normal(TRV[v]["cls"]) else 1 for v in VAL]
BATCH = 30
ITERS = 3000
torch.manual_seed(0)
net = MIL(D).to(dev)
net.train()
opt = torch.optim.Adagrad(net.parameters(), lr=0.001, weight_decay=1e-3)
best = (-1, None, 0)
hist = []
t0 = time.time()
for it in range(1, ITERS + 1):
    ba = random.sample(FIT_A, BATCH)
    bn = random.sample(FIT_N, BATCH)
    xa = torch.from_numpy(np.stack([Btr[v] for v in ba])).to(dev)
    xb = torch.from_numpy(np.stack([Btr[v] for v in bn])).to(dev)
    sa = net(xa)
    sb = net(xb)
    rank = torch.clamp(1.0 - sa.max(dim=1).values + sb.max(dim=1).values, min=0).mean()
    smooth = ((sa[:, 1:] - sa[:, :-1]) ** 2).sum(dim=1).mean()
    sparse = sa.sum(dim=1).mean()
    loss = rank + 8e-5 * smooth + 8e-5 * sparse
    opt.zero_grad(); loss.backward(); opt.step()
    if it % 100 == 0:
        a = video_level_auc(net, VAL, val_lab)
        hist.append({"iter": it, "loss": round(float(loss), 5), "val_video_auc": round(float(a), 5)})
        if a > best[0]:
            best = (a, {k: v.detach().clone() for k, v in net.state_dict().items()}, it)
        print("iter %4d loss %.4f  val video-level AUC %.4f  (best %.4f @ %d)  %.0fs"
              % (it, loss, a, best[0], best[2], time.time() - t0), flush=True)
print("selected iteration %d by held-out video-level AUC %.5f (test frames never used for selection)" % (best[2], best[0]))
net.load_state_dict(best[1])
net.eval()

In [ ]:
# ---- frame-level labels from the official annotations, then AUC ----
def frame_labels(v, idxs):
    ivs = ANNO[v]["intervals"]
    y = np.zeros(len(idxs), dtype=int)
    for k, i in enumerate(idxs):
        for s, e in ivs:
            if s <= i <= e:
                y[k] = 1
                break
    return y

# segment scores expanded back to frames (the literature protocol)
by_test = collections.defaultdict(list)
for i, v in enumerate(test_owner):
    by_test[v].append(i)

Y, S_seg, S_frame, vid_of = [], [], [], []
with torch.inference_mode():
    for v in te_keep:
        ii = np.array(sorted(by_test[v]))
        idxs = [test_fidx[i] for i in ii]
        y = frame_labels(v, idxs)
        seg = torch.from_numpy(Bte[v]).to(dev)
        ss = net(seg).cpu().numpy()                     # (32,)
        edges = np.linspace(0, len(ii), NSEG + 1).round().astype(int)
        exp = np.zeros(len(ii), dtype=np.float32)
        for k in range(NSEG):
            exp[edges[k]:edges[k + 1]] = ss[k]
        ff = Fte[ii].astype(np.float32)
        ff = ff / np.maximum(np.linalg.norm(ff, axis=1, keepdims=True), 1e-8)
        per = net(torch.from_numpy(ff.astype(np.float32)).to(dev)).cpu().numpy()
        Y.append(y); S_seg.append(exp); S_frame.append(per)
        vid_of += [v] * len(ii)
Y = np.concatenate(Y); S_seg = np.concatenate(S_seg); S_frame = np.concatenate(S_frame)
print("test frames scored:", len(Y), "| anomalous frames:", int(Y.sum()),
      "(%.3f%%)" % (100.0 * Y.mean()))
assert len(Y) >= 20000, "only %d test frames scored" % len(Y)
assert 0 < Y.sum() < len(Y), "frame labels are degenerate - annotation mapping is broken"
assert 0.01 < Y.mean() < 0.6, "anomalous frame fraction %.4f is implausible for UCF-Crime" % Y.mean()

auc_seg = float(roc_auc_score(Y, S_seg))
auc_frame = float(roc_auc_score(Y, S_frame))
print("FRAME-LEVEL AUC (32-segment scores expanded to frames) = %.5f" % auc_seg)
print("FRAME-LEVEL AUC (per-frame feature scored directly)    = %.5f" % auc_frame)
assert auc_seg > 0.5, "AUC at or below chance - the model learnt nothing, do not report this as a result"

def fpr_at_recall(y, s, target):
    fpr, tpr, thr = roc_curve(y, s)
    k = int(np.searchsorted(tpr, target, side="left"))
    k = min(k, len(tpr) - 1)
    return float(fpr[k]), float(tpr[k]), float(thr[k])

OP = {}
for r in [0.5, 0.8, 0.9]:
    f, t, th = fpr_at_recall(Y, S_seg, r)
    OP["recall_%.1f" % r] = {"fpr": round(f, 5), "actual_tpr": round(t, 5), "threshold": round(th, 5)}
    print("at recall %.1f : FPR = %.5f (threshold %.4f)" % (r, f, th))

# per-class AUC (anomalous frames of that class vs all normal-video frames)
vid_of = np.array(vid_of)
norm_mask = np.array([len(ANNO[v]["intervals"]) == 0 for v in vid_of])
per_class_auc = {}
for cls in sorted(set(TEV[v]["cls"] for v in te_keep)):
    if is_normal(cls):
        continue
    m = np.array([TEV[v]["cls"] == cls for v in vid_of]) | norm_mask
    if Y[m].sum() > 20 and (Y[m] == 0).sum() > 20:
        per_class_auc[cls] = round(float(roc_auc_score(Y[m], S_seg[m])), 5)
print("per-class frame AUC (class anomalous frames vs all normal-video frames):")
for k, v in sorted(per_class_auc.items(), key=lambda kv: kv[1]):
    print("   %-20s %.4f" % (k, v))

In [ ]:
# ---- the naive baseline: pretend every frame of an anomalous video is anomalous ----
from sklearn.linear_model import LogisticRegression
Xn, yn = [], []
for v in tr_keep:
    if v in Btr:
        Xn.append(Btr[v])
        yn += [0 if is_normal(TRV[v]["cls"]) else 1] * NSEG
Xn = np.concatenate(Xn)
yn = np.array(yn)
print("naive training segments:", Xn.shape, "positives:", int(yn.sum()))
lr = LogisticRegression(C=1.0, max_iter=2000)
lr.fit(Xn, yn)
S_naive = []
for v in te_keep:
    ii = np.array(sorted(by_test[v]))
    p = lr.predict_proba(Bte[v])[:, 1]
    edges = np.linspace(0, len(ii), NSEG + 1).round().astype(int)
    e = np.zeros(len(ii), dtype=np.float32)
    for k in range(NSEG):
        e[edges[k]:edges[k + 1]] = p[k]
    S_naive.append(e)
S_naive = np.concatenate(S_naive)
auc_naive = float(roc_auc_score(Y, S_naive))
print("naive full-video-label classifier, frame-level AUC = %.5f" % auc_naive)
print("MIL advantage over naive: %+.5f" % (auc_seg - auc_naive))

In [ ]:
# ---- BAR CHECK: judge this run against the published number for the same
# ---- model on the same benchmark. A number far below the bar is a bug
# ---- report about this pipeline, not a finding about the model.
BAR_PASS, BAR_BUG, BAR_REF = 0.70, 0.60, "Sultani et al. 2018 report ~0.754 frame-level AUC on UCF-Crime"
got = auc_seg
verdict = "PASS" if got >= BAR_PASS else ("BUG - DIAGNOSE, DO NOT PUBLISH" if got < BAR_BUG else "BELOW BAR")
print("=" * 64)
print("BAR: frame-level AUC >= %.2f   (reference: %s)" % (BAR_PASS, BAR_REF))
print("GOT: frame-level AUC  = %.5f" % got)
print("VERDICT: %s" % verdict)
if got < 0.55:
    print("An AUC this close to 0.50 means the model learnt nothing. It must NOT be written up as a result.")
print("=" * 64)

out = {
    "task": "UCF-Crime surveillance anomaly detection, frame-level ROC AUC",
    "method": "weakly-supervised MIL ranking (Sultani-style) on frozen ImageNet ResNet-50 frame features; 32 segments per video",
    "backbone": "torchvision resnet50, ResNet50_Weights.IMAGENET1K_V2, frozen, 2048-d",
    "supervision_at_train": "VIDEO-level labels only (anomalous vs normal), from the Train/ class folders",
    "labels_at_test": "official UCF-Crime per-frame temporal annotations",
    "annotation_source_url": SRC,
    "frame_dataset": "Kaggle UCF-Crime extracted frames; frame index parsed from the filename, which is the original video frame number",
    "frame_size": list(im0.size),
    "n_train_videos": len(tr_keep),
    "n_train_frames_embedded": int(len(train_paths)),
    "max_train_frames_per_video": MAX_TRAIN_FRAMES_PER_VIDEO,
    "n_test_videos": len(te_keep),
    "n_test_frames_scored": int(len(Y)),
    "n_test_anomalous_frames": int(Y.sum()),
    "anomalous_frame_fraction": round(float(Y.mean()), 5),
    "train_test_video_overlap": len(overlap),
    "model_selection": {"criterion": "video-level AUC on train videos held out from the fit",
                        "selected_iter": best[2], "val_video_auc": round(float(best[0]), 5),
                        "note": "test frames were never used for selection"},
    "train_history": hist,
    "held_out_frame_level_auc": round(auc_seg, 5),
    "held_out_frame_level_auc_per_frame_scoring": round(auc_frame, 5),
    "operating_points": OP,
    "per_class_frame_auc": per_class_auc,
    "naive_baseline_frame_level_auc": round(auc_naive, 5),
    "naive_baseline_definition": "logistic regression trained as if every segment of an anomalous video were anomalous - the mistake weak supervision exists to avoid",
    "mil_minus_naive": round(auc_seg - auc_naive, 5),
    "this_is_not_a_theft_detector": [
        "This is a benchmark measurement, not a deployable system, and must not be pointed at a real person or place.",
        "UCF-Crime videos were selected and trimmed by annotators; its normal videos are not a sample of real surveillance footage, so the AUC does not transfer to a live camera.",
        "AUC is a ranking summary over a pooled frame set. The operating-point table is the honest view: see the false-positive rate required to reach each recall.",
        "An anomaly score is not a crime classification. The per-class AUCs above are computed from the dataset's own class folders and say nothing about intent, legality or identity.",
        "No face, person or identity is detected, tracked or recognised anywhere in this notebook.",
    ],
    "what_this_does_not_show": [
        "One feature extractor and one MIL head on one benchmark. It establishes no ranking against published UCF-Crime methods, which use different features (C3D/I3D) and the full frame rate.",
        "The frames in this Kaggle copy are decimated and small, so this number is not comparable to papers that use every frame at full resolution.",
        "The naive baseline comparison is a comparison of two things trained here, under identical features and splits; that part is a controlled contrast.",
    ],
}
json.dump(out, open("/kaggle/working/results.json", "w"), indent=1)
print(json.dumps(out, indent=1)[:6000])
print("\nWROTE /kaggle/working/results.json")